In [ ]:
import pandas as pd
import numpy as np


In [ ]:
tot_data = pd.read_csv("../data/select_index_data_cleaned_v2.csv")


In [ ]:
future_data = pd.read_csv("../data/futures_data_v1.csv")


In [ ]:
rename_columns = {"prccd" : "close", "prchd" : "high", "prcld" : "low", "tic_map" : "ticker", "tic" : "tic_code", "datadate":"date"}
future_rename_columns = {"Close" : "close", "High" : "high", "Low" : "low", "tic_map" : "ticker", "Symbol" : "tic_code", "Date":"date"}


In [ ]:
tot_data = tot_data.rename(columns=rename_columns)


In [ ]:
future_data = future_data.rename(columns=future_rename_columns)


In [ ]:
def sma(data, period):
    return data.rolling(window=period).mean()

def ema(data, period):
    return data.ewm(span=period, adjust=False).mean()

def macd(data, short_period=12, long_period=26, signal_period=9):
    short_ema = ema(data, short_period)
    long_ema = ema(data, long_period)
    macd_line = short_ema - long_ema
    signal_line = ema(macd_line, signal_period)
    return signal_line

def bollinger_bands(data, period=20, std=2):
    sma_ = sma(data, period)
    std_dev = data.rolling(window=period).std()
    upper_band = sma_ + (std_dev * std)
    lower_band = sma_ - (std_dev * std)
    return pd.DataFrame({'boll_ub': upper_band, 'boll_lb': lower_band})

def rsi(data, period=14):
    delta = data.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=period).mean()
    epsilon = 1e-10
    rs = gain / (loss + epsilon)
    return 100 - (100 / (1 + rs))

def cci(high, low, close, period=14):
    tp = (high + low + close) / 3
    sma_tp = sma(tp, period)
    mad = abs(tp - sma_tp).rolling(window=period).mean()
    epsilon = 1e-10
    return (tp - sma_tp) / (0.015 * mad + epsilon)

def sma_rolling(data, price_col):
    data_copy = data.copy()
    for period in [30, 60, 220, 252]:
        data_copy[f'SMA_{period}'] = sma(data_copy[price_col], period)
    return data_copy[[f'SMA_{p}' for p in [30, 60, 220, 252]]]

def compute_technical_indicators(
    data_df,
    price_col='close',
    high_col='high',
    low_col='low',
    ticker_col='ticker'
):
    data_df = data_df.copy()

    concat_data = pd.DataFrame()

    for tic in data_df[ticker_col].unique():
        imp = data_df[data_df[ticker_col] == tic].copy().reset_index(drop=True)

        macd_data = pd.DataFrame({'macd': macd(imp[price_col].shift(1))})
        boll_band = bollinger_bands(imp[price_col].shift(1))
        rsi_data = pd.DataFrame({'rsi': rsi(imp[price_col].shift(1))})
        cci_data = pd.DataFrame({'cci': cci(imp[high_col], imp[low_col], imp[price_col]).shift(1)})
        sma_data = sma_rolling(imp, price_col=price_col).shift(1)

        combined = pd.concat([imp, sma_data, macd_data, boll_band, rsi_data, cci_data], axis=1)
        concat_data = pd.concat([concat_data, combined], axis=0)

    concat_data.reset_index(drop=True, inplace=True)
    return concat_data


In [ ]:
# 만약 'adjust_close', 'adjust_high', 'adjust_low' 컬럼을 사용한다면:
indicator_df = compute_technical_indicators(
    tot_data,
    price_col='close',
    high_col='high',
    low_col='low',
    ticker_col='ticker'
)


In [ ]:
future_indicator_df = compute_technical_indicators(
    future_data,
    price_col='close',
    high_col='high',
    low_col='low',
    ticker_col='ticker'
)


In [ ]:
future_indicator_df.columns


## 분포 시각화 코드

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 지표 목록
indicators = ['SMA_30', 'SMA_60', 'SMA_220', 'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci']

# 전체 티커 목록
all_tickers = indicator_df['ticker'].unique().tolist()

# 시각화: 모든 티커별 지표 분포
fig, axes = plt.subplots(len(all_tickers), len(indicators), figsize=(24, 4 * len(all_tickers)))

for i, ticker in enumerate(all_tickers):
    ticker_df = indicator_df[indicator_df['ticker'] == ticker]
    for j, ind in enumerate(indicators):
        ax = axes[i][j] if len(all_tickers) > 1 else axes[j]
        sns.histplot(ticker_df[ind].dropna(), kde=True, ax=ax)
        ax.set_title(f'{ticker} - {ind}', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 지표 목록
indicators = ['SMA_30', 'SMA_60', 'SMA_220', 'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci']

# 전체 티커 목록
all_tickers = future_indicator_df['ticker'].unique().tolist()

# 시각화: 모든 티커별 지표 분포
fig, axes = plt.subplots(len(all_tickers), len(indicators), figsize=(24, 4 * len(all_tickers)))

for i, ticker in enumerate(all_tickers):
    ticker_df = future_indicator_df[future_indicator_df['ticker'] == ticker]
    for j, ind in enumerate(indicators):
        ax = axes[i][j] if len(all_tickers) > 1 else axes[j]
        sns.histplot(ticker_df[ind].dropna(), kde=True, ax=ax)
        ax.set_title(f'{ticker} - {ind}', fontsize=10)
        ax.set_xlabel('')
        ax.set_ylabel('')

plt.tight_layout()
plt.show()


## Slicing

In [ ]:
slicing_df = indicator_df[['date', "ticker", 'SMA_30', 'SMA_60', 'SMA_220', 'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci']]


In [ ]:
for ticker in slicing_df['ticker'].unique():
    ticker_df = slicing_df[slicing_df['ticker'] == ticker]
    print(ticker)
    print(ticker_df.isna().sum())


In [ ]:
future_slicing_df = future_indicator_df[['date', "ticker", 'SMA_30', 'SMA_60', 'SMA_220', 'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci']]


In [ ]:
for ticker in future_slicing_df['ticker'].unique():
    ticker_df = future_slicing_df[future_slicing_df['ticker'] == ticker]
    print(ticker)
    print(ticker_df.isna().sum())


## Scale

In [ ]:
import pandas as pd

# 데이터프레임이 df라고 가정
tot_data = tot_data.sort_values(['ticker', 'date'])  # 티커별, 날짜별 정렬
tot_data['ret'] = tot_data.groupby('ticker')['close'].pct_change().fillna(0)


In [ ]:
future_data = future_data.sort_values(['ticker', 'date'])  # 티커별, 날짜별 정렬
future_data['ret'] = future_data.groupby('ticker')['close'].pct_change().fillna(0)


In [ ]:
tot_data.sort_values(['date', 'ticker'], inplace=True)
tot_data.reset_index(drop=True, inplace=True)


In [ ]:
future_data.sort_values(['date', 'ticker'], inplace=True)
future_data.reset_index(drop=True, inplace=True)


In [ ]:
future_data[future_data["tic_code"]=="CC=F"]


In [ ]:
def _adjust_ohl(data):
    ###
    # Adjust the OHL price with the ratio
    ###
    data["adjust_high"] = data["high_ratio"] * data["adjust_close"]
    data["adjust_low"] = data["low_ratio"] * data["adjust_close"]
    return data

def scale_price(data):
    ###
    # Scale the price data using vectorized operations
    ###
    data = data.copy()  # 원본 데이터 보호
    data["adjust_close"] = data.groupby("ticker")["ret"].transform(lambda x: (1 + x).cumprod())
    data = _adjust_ohl(data)
    return data


def cal_ratio(total_data):
    ###
    # Calculate the ratio of OHL price to close price
    ###
    total_data["high_ratio"] = total_data["high"] / total_data["close"]
    total_data["low_ratio"] = total_data["low"] / total_data["close"]
    return total_data


In [ ]:
# 데이터 전처리
scale_df = cal_ratio(tot_data)
scale_df = scale_price(scale_df)


In [ ]:
future_scale_df = cal_ratio(future_data)
future_scale_df = scale_price(future_scale_df)


In [ ]:
def close_and_return_momentum(data):
    data_imp = data.copy()
    periods = [20, 40, 60, 80, 100, 120, 140, 160, 180, 220, 240, 252]

    columns_list = ["return_1m", "return_3m", "return_6m", "return_12m", "return_avg", "mom_12m", "mom_score"]

    for idx, period in enumerate(periods):
        col_name = f'close_{idx+1}_month'
        data_imp[col_name] = data_imp['adjust_close'].shift(period)
        columns_list.append(col_name)

    # 모멘텀 및 수익률 계산
    data_imp['return_1m'] = (data_imp['adjust_close'] - data_imp['close_1_month']) / data_imp['close_1_month']
    data_imp['return_3m'] = (data_imp['adjust_close'] - data_imp['close_3_month']) / data_imp['close_3_month']
    data_imp['return_6m'] = (data_imp['adjust_close'] - data_imp['close_6_month']) / data_imp['close_6_month']
    data_imp['return_12m'] = (data_imp['adjust_close'] - data_imp['close_12_month']) / data_imp['close_12_month']
    data_imp['return_avg'] = data_imp[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    data_imp["mom_12m"] = data_imp["adjust_close"] / data_imp["close_12_month"] - 1
    data_imp["mom_score"] = (
        12 * data_imp["return_1m"] +
        4 * data_imp["return_3m"] +
        2 * data_imp["return_6m"] +
        1 * data_imp["return_12m"]
    ) / 19

    return data_imp[columns_list]


def add_momentum_features(data_df):
    """
    티커별로 모멘텀 리턴 및 스코어 계산 후 원본 데이터와 결합

    Args:
        data_df (pd.DataFrame): 'adjust_close'와 'tic' 포함된 데이터

    Returns:
        pd.DataFrame: 모멘텀 피처가 추가된 전체 데이터
    """
    result_df = pd.DataFrame()

    for ticker in data_df["ticker"].unique():
        imp = data_df[data_df["ticker"] == ticker].copy().reset_index(drop=True)
        momentum_df = close_and_return_momentum(imp)
        combined = pd.concat([imp, momentum_df], axis=1)
        result_df = pd.concat([result_df, combined], axis=0)

    result_df.reset_index(drop=True, inplace=True)
    return result_df


In [ ]:
def _adjust_ohl(data):
    ###
    # Adjust the OHL price with the ratio
    ###
    data["adjust_high"] = data["high_ratio"] * data["adjust_close"]
    data["adjust_low"] = data["low_ratio"] * data["adjust_close"]
    return data

def scale_price(data):
    ###
    # Scale the price data using vectorized operations
    ###
    data = data.copy()  # 원본 데이터 보호
    data["adjust_close"] = data.groupby("ticker")["ret"].transform(lambda x: (1 + x).cumprod())
    data = _adjust_ohl(data)
    return data


def cal_ratio(total_data):
    ###
    # Calculate the ratio of OHL price to close price
    ###
    total_data["high_ratio"] = total_data["high"] / total_data["close"]
    total_data["low_ratio"] = total_data["low"] / total_data["close"]
    return total_data


In [ ]:
scale_df = add_momentum_features(scale_df)


In [ ]:
future_scale_df = add_momentum_features(future_scale_df)


In [ ]:
tot_df = scale_df.merge(slicing_df, on = ["date", "ticker"], how = "left")


In [ ]:
future_tot_df = future_scale_df.merge(future_slicing_df, on = ["date", "ticker"], how = "left")


In [ ]:
# indicator_cols = [
#     'SMA_30', 'SMA_60', 'SMA_220', 'SMA_252',
#     'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci',
#     'return_1m', 'return_3m', 'return_6m', 'return_12m',
#     'return_avg', 'mom_12m', 'mom_score'
# ]

# # 위 컬럼 중 하나라도 NaN이 있으면 제거
# clean_df = tot_df.dropna(subset=indicator_cols).reset_index(drop=True)


In [ ]:
clean_df = tot_df


In [ ]:
clean_future_df = future_tot_df


In [ ]:
clean_df.sort_values(['date', 'ticker'], inplace=True)
clean_df.reset_index(drop=True, inplace=True)


In [ ]:
clean_future_df.sort_values(['date', 'ticker'], inplace=True)
clean_future_df.reset_index(drop=True, inplace=True)


In [ ]:
# for tic in clean_df.ticker.unique():
#     tic_df = clean_df[clean_df.ticker == tic]
#     print(tic)
#     print(tic_df.date.min(), tic_df.date.max())


In [ ]:
clean_df["paa_score"] = clean_df.apply(lambda row: (row["close"] / row["SMA_252"]) - 1, axis=1)
clean_df["gtaa_buy_signal"] = (clean_df["close"] > clean_df["SMA_220"]).astype(int)
clean_df["paa_buy_signal"] = (clean_df["paa_score"] > 0).astype(int)
clean_df["daa_buy_signal"] = (clean_df["mom_score"] > 0).astype(int)


In [ ]:
clean_future_df["paa_score"] = clean_future_df.apply(lambda row: (row["close"] / row["SMA_252"]) - 1, axis=1)
clean_future_df["gtaa_buy_signal"] = (clean_future_df["close"] > clean_future_df["SMA_220"]).astype(int)
clean_future_df["paa_buy_signal"] = (clean_future_df["paa_score"] > 0).astype(int)
clean_future_df["daa_buy_signal"] = (clean_future_df["mom_score"] > 0).astype(int)


In [ ]:
clean_future_df["paa_score"] 


In [ ]:
import numpy as np

def cumulative_momentum_product(row, months=12, risk_free_rate=0.0):
    """
    특정 행(row)에 대해 최근 N개월 동안의 개별 월별 모멘텀을 누적 곱한 후,
    risk-free rate(무위험 이자율)과 비교하여 투자 여부 결정.

    Parameters:
        row (pd.Series): 개별 행 데이터
        months (int): 계산할 개월 수 (기본값: 12)
        risk_free_rate (float): 무위험 이자율 (예: 1개월 미국 국채 금리)

    Returns:
        float: 누적 모멘텀 수익률 또는 risk-free rate
    """
    # 최근 N개월 동안의 월별 수익률을 기반으로 누적 수익률 계산
    monthly_returns = [(row[f"close_{t}_month"] / row[f"close_{t+1}_month"]) - 1
                       for t in range(1, months) if f"close_{t}_month" in row and f"close_{t+1}_month" in row]

    # NaN 제거 (데이터 누락 방지)
    
    # 누적 수익률 계산: ∏ (1 + R_t)
    cumulative_return = np.prod([1 + m for m in monthly_returns])

    # 투자 결정: 누적 수익률이 risk-free rate보다 크면 투자, 아니면 risk-free rate 적용
    return cumulative_return if cumulative_return > risk_free_rate else risk_free_rate

# DataFrame에 apply 적용


In [ ]:
clean_df["momentum_product"] = clean_df.apply(lambda row: cumulative_momentum_product(row, months=12), axis=1)


In [ ]:
clean_future_df["momentum_product"] = clean_future_df.apply(lambda row: cumulative_momentum_product(row, months=12), axis=1)


In [ ]:
clean_df["dm_buy_signal"] = (clean_df["momentum_product"] > 1).astype(int)


In [ ]:
clean_future_df["dm_buy_signal"] = (clean_future_df["momentum_product"] > 1).astype(int)


In [ ]:
# clean_df.date = pd.to_datetime(clean_df.date)


In [ ]:
# clean_df['date'] = clean_df['date'].dt.strftime('%Y-%m-%d')


In [ ]:
clean_df.to_csv("../data/index_preprocess_data_v1.csv", index=False)


In [ ]:
clean_future_df.to_csv("../data/futures_preprocess_data_v1.csv", index=False)


In [ ]:
import pandas as pd


In [ ]:
clean_df = pd.read_csv("../data/index_preprocess_data_v1.csv")
clean_future_df = pd.read_csv("../data/futures_preprocess_data_v1.csv")


In [ ]:
clean_future_df["date"].unique()[252]


In [ ]:
clean_future_df["date"].unique()[0]


In [ ]:
clean_df["date"].unique()[252]


In [ ]:
TRAIN_START_DATE = '2003-02-19'
TRAIN_END_DATE = '2013-12-31'
VALID_START_DATE = '2014-01-01'
VALID_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2024-12-31'


In [ ]:
def data_split(df, start, end, target_date_col="date"):
    """
    split the dataset into training or testing using date
    :param data: (df) pandas dataframe, start, end
    :return: (df) pandas dataframe
    """
    data = df[(df[target_date_col] >= start) & (df[target_date_col] <= end)]
    data = data.sort_values([target_date_col, "ticker"], ignore_index=True)
    # data.index = data[target_date_col].factorize()[0]
    return data


In [ ]:
before_df = clean_df[(clean_df["date"] < TRAIN_START_DATE)]
before_df = before_df.sort_values(["date", "ticker"], ignore_index=True)


In [ ]:
before_future_df = clean_future_df[(clean_future_df["date"] < TRAIN_START_DATE)]
before_future_df = before_future_df.sort_values(["date", "ticker"], ignore_index=True)


In [ ]:
train = data_split(clean_df, TRAIN_START_DATE,TRAIN_END_DATE)
test = data_split(clean_df, VALID_START_DATE, TEST_END_DATE)


In [ ]:
future_train = data_split(clean_future_df, TRAIN_START_DATE,TRAIN_END_DATE)
future_test = data_split(clean_future_df, VALID_START_DATE, TEST_END_DATE)


In [ ]:
def min_max_normalize_by_ticker_train_test(train_df, test_df, columns):
    """
    주어진 컬럼들을 train 기준으로 티커별 min-max 정규화
    
    Parameters:
        train_df (DataFrame): 학습용 데이터
        test_df (DataFrame): 테스트용 데이터
        columns (list): 정규화할 컬럼 리스트
        
    Returns:
        train_df, test_df: 정규화된 결과가 포함된 데이터프레임
    """
    # 티커별로 정규화 통계 계산
    stats = train_df.groupby("ticker")[columns].agg(["min", "max"])
    
    # 컬럼명 정리 (MultiIndex → flat column name)
    stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]

    # train/test에 붙이기
    train_df = train_df.merge(stats, on="ticker", how="left")
    test_df = test_df.merge(stats, on="ticker", how="left")

    # 컬럼별 정규화 수행
    for col in columns:
        min_col = f"{col}_min"
        max_col = f"{col}_max"
        norm_col = f"{col}_norm"

        train_df[norm_col] = (train_df[col] - train_df[min_col]) / (train_df[max_col] - train_df[min_col])
        test_df[norm_col] = (test_df[col] - test_df[min_col]) / (test_df[max_col] - test_df[min_col])

    # 불필요한 min/max 컬럼 제거
    cols_to_drop = [f"{col}_min" for col in columns] + [f"{col}_max" for col in columns]
    train_df.drop(columns=cols_to_drop, inplace=True)
    test_df.drop(columns=cols_to_drop, inplace=True)

    return train_df, test_df


In [ ]:
columns_to_normalize = [ 'adjust_close', 'return_1m', 'return_3m', 'return_6m', 'return_12m', 'return_avg',
       'mom_12m', 'mom_score', 'SMA_30', 'SMA_60', 'SMA_220', 'SMA_252',
                        'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci']

train_df, test_df = min_max_normalize_by_ticker_train_test(train, test, columns_to_normalize)


In [ ]:
future_train_df, future_test_df = min_max_normalize_by_ticker_train_test(future_train, future_test, columns_to_normalize)


In [ ]:
before_df.to_csv("../data/index_lookback.csv")
train_df.to_csv("../data/index_train.csv")
test_df.to_csv("../data/index_test.csv")


In [ ]:
before_future_df.to_csv("../data/future_lookback.csv")
future_train_df.to_csv("../data/future_train.csv")
future_test_df.to_csv("../data/future_test.csv")
